# SQL SELECT Queries

In [1]:
import pandas as pd
import numpy as np
import polars as pl

In [2]:
df_customers = pd.read_csv('../data/sales_customers.csv')
df_employees = pd.read_csv('../data/sales_employees.csv')
df_orders = pd.read_csv('../data/sales_orders.csv')
df_orderarchive = pd.read_csv('../data/sales_ordersarchive.csv')
df_products = pd.read_csv('../data/sales_products.csv')

pl_customers = pl.read_csv('../data/sales_customers.csv')
pl_employees = pl.read_csv('../data/sales_employees.csv')
pl_orders = pl.read_csv('../data/sales_orders.csv')
pl_orderarchive = pl.read_csv('../data/sales_ordersarchive.csv')
pl_products = pl.read_csv('../data/sales_products.csv')

### Retrieve each customer's name, country and score

In [10]:
df_customers[['firstname','country','score']]

,firstname,country,score
0,Jossef,Germany,350.0
1,Kevin,USA,900.0
2,Mary,USA,750.0
3,Mark,Germany,500.0
4,Anna,USA,NaN


In [15]:
pl_customers.select([
    'firstname', 'country', 'score'
]).with_row_index(name='id')

id,firstname,country,score
u32,str,str,i64
0,"""Jossef""","""Germany""",350
1,"""Kevin""","""USA""",900
2,"""Mary""","""USA""",750
3,"""Mark""","""Germany""",500
4,"""Anna""","""USA""",null


## WHERE - Filters your data based on a Condition

### Retrieve customers with a score not equal to 0 and is not null

In [28]:
df_customers[
    (df_customers['score'] != 0) &
    (df_customers['score'].notnull()) &
    (df_customers['country'].str.lower().str.strip() == 'germany')
].reset_index(drop=True)

,customerid,firstname,lastname,country,score
0,1,Jossef,Goldberg,Germany,350.0
1,4,Mark,Schwarz,Germany,500.0


In [27]:
pl_customers.filter(
    (pl.col('score') != 0) &
    (pl.col('score').is_not_null()) &
    (pl.col('country').str.to_lowercase().str.strip_chars() == 'germany')
).with_row_index(name='id')

id,customerid,firstname,lastname,country,score
u32,i64,str,str,str,i64
0,1,"""Jossef""","""Goldberg""","""Germany""",350
1,4,"""Mark""","""Schwarz""","""Germany""",500


### ORDER BY

#### Retrieve all customers and sort the results by the hihest score first

In [33]:
df_customers.sort_values(by='score', ascending=False, na_position='last').reset_index(drop=True)

,customerid,firstname,lastname,country,score
0,2,Kevin,Brown,USA,900.0
1,3,Mary,NaN,USA,750.0
2,4,Mark,Schwarz,Germany,500.0
3,1,Jossef,Goldberg,Germany,350.0
4,5,Anna,Adams,USA,NaN


In [32]:
pl_customers.sort('score',descending=True, nulls_last=True).with_row_index()

index,customerid,firstname,lastname,country,score
u32,i64,str,str,str,i64
0,2,"""Kevin""","""Brown""","""USA""",900
1,3,"""Mary""",null,"""USA""",750
2,4,"""Mark""","""Schwarz""","""Germany""",500
3,1,"""Jossef""","""Goldberg""","""Germany""",350
4,5,"""Anna""","""Adams""","""USA""",null


#### Retrieve all customers and sort the results by the country and then bythe highest score

In [35]:
res_pd = (
    df_customers
    .sort_values(
        by=['country','score'],
        ascending=[True, False],
        na_position='last'
    )
    .reset_index(drop=True)
)

res_pd

,customerid,firstname,lastname,country,score
0,4,Mark,Schwarz,Germany,500.0
1,1,Jossef,Goldberg,Germany,350.0
2,2,Kevin,Brown,USA,900.0
3,3,Mary,NaN,USA,750.0
4,5,Anna,Adams,USA,NaN


In [36]:
res_pl = (
    pl_customers
    .sort(
        ['country','score'],
        descending=[False, True],
        nulls_last = True
    )
    .with_row_index(offset=1)
)

res_pl

index,customerid,firstname,lastname,country,score
u32,i64,str,str,str,i64
1,4,"""Mark""","""Schwarz""","""Germany""",500
2,1,"""Jossef""","""Goldberg""","""Germany""",350
3,2,"""Kevin""","""Brown""","""USA""",900
4,3,"""Mary""",null,"""USA""",750
5,5,"""Anna""","""Adams""","""USA""",null


#### Find the total score for each country

In [44]:
res_pd = df_customers.groupby('country')['score'].agg(total_score='sum').reset_index()

res_pd

,country,total_score
0,Germany,850.0
1,USA,1650.0


In [49]:
res_pl = pl_customers.group_by('country').agg(
    pl.col('score').sum().alias('total_score'),
    pl.len().alias('total_customers')
).with_row_index()

res_pl


index,country,total_score,total_customers
u32,str,i64,u32
0,"""Germany""",850,2
1,"""USA""",1650,3


### HAVING

#### Find the average score for each country considering only customers with a score is null And return only those countries with an average score greater than 430

In [51]:
res_pd = (
    df_customers
    .assign(score=df_customers['score'].fillna(0))
    .groupby('country')['score']
    .mean()
    .reset_index()
)

res_pd = res_pd[res_pd['score'] > 430]

res_pd

,country,score
1,USA,550.0


In [52]:
res_pl = (
    pl_customers
    .with_columns(pl.col('score').fill_null(0))
    .group_by('country')
    .agg(
        pl.col('score').mean().alias('avg_score')
    )
    .filter(pl.col('avg_score') > 430)
)

res_pl

country,avg_score
str,f64
"""USA""",550.0
